# DocTR · Kaggle

Kaggle CPU. Enable Internet. Run all cells for a 10-page pilot. For all 9,000 pages set MODE to full. Select a contiguous FULL_SLICE for shorter sessions. Successful pages are committed individually. On Kaggle save both the notebook version and its SQLite outputs, then restore the latest worker files to /kaggle/working/oprb_results before resuming.

Runtime repair: `wrapt` is installed inside the private environment before torch/docTR. The private kernel, weight preparation and inference workers use the same sanitized environment and isolated interpreter (`-I`). A startup/import/torchvision-operator check runs before the data download. Subprocess output is streamed into the notebook; Python tracebacks and download errors remain visible.

After importing this revision, restart the Kaggle session and run all cells from the top with Internet enabled, Accelerator None (CPU), and MODE="pilot". Do not rerun the inference cell while another run is active. OCR backbones/options, seed-42 pages, per-page timing, normalization and scoring are unchanged. Runtime package changes may create a new checkpoint hash; retain older SQLite files separately.


In [ ]:
from pathlib import Path
import sys, os, subprocess, tempfile, json
MODE = "pilot"
FULL_SLICE = slice(None)
if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Open this notebook on Kaggle.")
OUTPUT = Path("/kaggle/working/oprb_results")
GPU_WORKERS = 1
# Keep Kaggle's startup hooks and user packages out of every private process.
runtime_env = {k: v for k, v in os.environ.items()
               if k not in {"PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE", "PYTHONINSPECT"}}
runtime_env.update(PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1", USE_TF="0", USE_TORCH="1",
                   CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="2", OMP_THREAD_LIMIT="2")


def run_logged(args, **kwargs):
    """Forward subprocess output to this notebook and stop children on interruption."""
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", bufsize=1, **kwargs)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        returncode = proc.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, args)
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        proc.stdout.close()


import venv
from jupyter_client import KernelManager
from IPython.core.magic import register_cell_magic
runtime = Path(tempfile.gettempdir()) / ('oprb-private-doctr' + "-" + sys.implementation.cache_tag)
python = runtime / "bin/python"
if not python.exists():
    venv.create(runtime, with_pip=False)
pip = [sys.executable, "-I", "-m", "pip", "--python", str(python)]
# Install the missing startup-hook dependency inside the model venv, not globally.
run_logged(pip + ["install", "wrapt>=1.17,<3"], env=runtime_env)
run_logged(pip + ["install", "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.11.0+cpu", "torchvision==0.26.0+cpu"], env=runtime_env)
requirements = ['jiwer==4.0.0', 'pandas>=2,<4', 'ipykernel>=6,<8', 'pillow', 'torch==2.11.0+cpu', 'torchvision==0.26.0+cpu', 'python-doctr==1.0.1', 'huggingface-hub>=0.20,<2', 'wrapt>=1.17,<3']
run_logged(pip + ["install", *requirements], env=runtime_env)
run_logged(pip + ["check"], env=runtime_env)
# Check startup hooks and the torch/torchvision CPU binary pair before downloading OPRB.
probe_code = """import sys, wrapt, torch, torchvision, doctr
from torchvision.ops import nms
torch.set_num_threads(2)
assert torch.version.cuda is None, "Expected the pinned CPU build of PyTorch"
assert len(nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), 0.5)) == 1
print("Runtime OK:", sys.version.split()[0], "torch=" + torch.__version__,
      "torchvision=" + torchvision.__version__, "doctr=" + doctr.__version__, flush=True)
"""
probe = subprocess.run([str(python), "-I", "-u", "-c", probe_code], env=runtime_env,
                       capture_output=True, text=True, encoding="utf-8")
print(probe.stdout, end="", flush=True)
if probe.stderr:
    print(probe.stderr, end="", flush=True)
probe.check_returncode()
if "Error in sitecustomize" in probe.stderr:
    raise RuntimeError("The private interpreter still has a failing startup hook; inspect the diagnostic above.")
if "manager" in globals() and manager.has_kernel:
    client.stop_channels()
    manager.shutdown_kernel(now=True)
manager = KernelManager(kernel_name="python3")
manager.kernel_spec.argv = [str(python), "-I", "-m", "ipykernel_launcher", "-f", "{connection_file}"]
manager.start_kernel(env=runtime_env)
client = manager.blocking_client()
client.start_channels()
client.wait_for_ready(timeout=120)

def show_output(message):
    from IPython.display import display
    content = message["content"]
    if message["msg_type"] == "stream":
        print(content["text"], end="", flush=True)
    elif message["msg_type"] in {"display_data", "execute_result"}:
        display(content["data"], raw=True)
    elif message["msg_type"] == "error":
        print("\n".join(content["traceback"]), flush=True)

@register_cell_magic
def benchmark(line, cell):
    try:
        reply = client.execute_interactive(cell, timeout=None, output_hook=show_output)
    except KeyboardInterrupt:
        manager.interrupt_kernel()
        raise
    if reply["content"]["status"] != "ok":
        raise RuntimeError(reply["content"].get("evalue", "Private kernel failed"))

benchmark("", f"from pathlib import Path\nimport os, sys, subprocess, tempfile, json\n"
          f"assert sys.prefix == {str(runtime)!r}\nMODE={MODE!r}\nFULL_SLICE={FULL_SLICE!r}\n"
          f"OUTPUT=Path({str(OUTPUT)!r})\nGPU_WORKERS={GPU_WORKERS}")
benchmark("", "runtime_env = dict(os.environ)")
import inspect
benchmark("", inspect.getsource(run_logged))


In [ ]:
%%benchmark
from doctr.models import ocr_predictor
MODEL = 'doctr'
IS_GPU = False
model_python = sys.executable
OUTPUT.mkdir(parents=True, exist_ok=True)
assert MODE in {"pilot","full"}
CACHE = Path(tempfile.gettempdir()) / "oprb_test"
import hashlib, json, random, shutil, sqlite3, tarfile, time, unicodedata
from collections import Counter
from importlib.metadata import version
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from PIL import Image
from jiwer import cer, wer
import pandas as pd

REPO = "kpurkayastha/OPRB"
DATA_REV = 'c7ab9fdf2bf33f297bba62e4842d3e5094d11ec2'
ROOT = CACHE / DATA_REV
COUNTS = {"text": 6000, "nontext": 3000}
pages = []
for region, count in COUNTS.items():
    for kind, suffix in [("img", ".jpg"), ("gt", ".txt")]:
        target = ROOT / region / kind
        target.mkdir(parents=True, exist_ok=True)
        if not (target / ".complete").is_file():
            archive = hf_hub_download(REPO, f"test/{region}/{kind}.tar.gz", repo_type="dataset", revision=DATA_REV)
            seen = set()
            with tarfile.open(archive, "r:gz") as tar:
                for member in tar:
                    name = Path(member.name).name
                    if not member.isfile() or Path(name).suffix.lower() != suffix:
                        continue
                    if name in seen:
                        raise ValueError(f"Repeated basename: {name}")
                    seen.add(name)
                    with tar.extractfile(member) as src, (target / name).open("wb") as dest:
                        shutil.copyfileobj(src, dest)
            assert len(seen) == count, (region, kind, len(seen))
            (target / ".complete").touch()
    imgs = {p.stem for p in (ROOT / region / "img").glob("*.jpg")}
    refs = {p.stem for p in (ROOT / region / "gt").glob("*.txt")}
    assert imgs == refs and len(imgs) == count, (region, len(imgs), len(refs))
    pages.extend((region, stem) for stem in sorted(imgs))
selected = pages[FULL_SLICE] if MODE == "full" else [
    key for region in COUNTS
    for key in random.Random(42).sample([p for p in pages if p[0] == region], 5)
]
print(DATA_REV, Counter(r for r, _ in selected))

In [ ]:
%%benchmark
def reference(region, stem):
    tokens = []
    path = ROOT / region / "gt" / (stem + ".txt")
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        fields = line.split("\t")
        if len(fields) not in (10, 11):
            raise ValueError(f"Malformed GT row in {path.name}: {line[:60]!r}")
        if fields[0].strip():
            tokens.append(fields[0].strip())
    if not tokens:
        raise ValueError(f"Empty GT: {path.name}")
    return " ".join(tokens)

def normalize(s):
    return " ".join(unicodedata.normalize("NFKC", s).casefold().split())

def metrics(ref, hyp):
    ref, hyp = normalize(ref), normalize(hyp)
    a, b = Counter(ref.split()), Counter(hyp.split())
    hits = sum((a & b).values())
    p = hits / sum(b.values()) if b else 0
    r = hits / sum(a.values())
    return cer(ref, hyp), wer(ref, hyp), p, r, 2*p*r/(p+r) if hits else 0

assert metrics("Ａ b", "a B") == (0, 0, 1, 1, 1)
assert metrics("a", "") == (1, 1, 0, 0, 0)
assert metrics("a a", "a")[2:] == (1, .5, 2/3)
assert metrics("a b", "b a")[1:] == (1, 1, 1, 1)

In [ ]:
%%benchmark
from pathlib import Path
weights = {}
revisions = {}

def load_reader(model, weights, prompt):
    import os
    os.environ["DOCTR_CACHE_DIR"] = weights["doctr"]
    from doctr.io import DocumentFile
    from doctr.models import ocr_predictor
    engine = ocr_predictor(det_arch="db_mobilenet_v3_large", reco_arch="crnn_mobilenet_v3_small",
                           pretrained=True, assume_straight_pages=True).eval()
    def read(path):
        document = engine(DocumentFile.from_images(str(path)))
        return document.render(), False
    return read

import inspect
weights["doctr"] = str(CACHE / "doctr-mobile-v1")
setup = inspect.getsource(load_reader) + "\nload_reader('doctr', " + repr(weights) + ", '')"
print("Preparing docTR MobileNet weights in an isolated CPU process...", flush=True)
run_logged([str(model_python), "-I", "-u", "-c", setup], env=runtime_env)
revisions = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
             for p in Path(weights["doctr"]).rglob("*.pt")}

weight_files = [p for folder in weights.values() for p in Path(folder).rglob("*")
                if p.suffix in {".safetensors", ".pdiparams", ".pt", ".pth", ".traineddata"}]
assert len(revisions) >= 2, "Both MobileNet detector and recognizer weights must be present"
assert weight_files, "Inspect weight filenames for this model"
weight_mib = sum(p.stat().st_size for p in weight_files) / 2**20

In [ ]:
%%benchmark
# Each worker owns one database. Re-running skips successful rows and retries errors.
import inspect, platform
from concurrent.futures import ThreadPoolExecutor

def evaluate(job):
    import hashlib, json, os, platform, sqlite3, time
    from pathlib import Path
    model, region_pages = job["model"], job["pages"]
    sync = lambda: None
    if job["gpu"]:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Select a CUDA GPU runtime or reduce GPU_WORKERS.")
        device = torch.cuda.get_device_name()
        sync = torch.cuda.synchronize
    else:
        device = "CPU"
    cfg = dict(job["config"], device=device)
    key = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()
    db = sqlite3.connect(Path(job["output"]) / f"{model}-{key[:16]}-{job['worker']}.sqlite")
    db.execute("CREATE TABLE IF NOT EXISTS config (json TEXT PRIMARY KEY)")
    db.execute("INSERT OR IGNORE INTO config VALUES (?)", (json.dumps(cfg, sort_keys=True),))
    db.execute("""CREATE TABLE IF NOT EXISTS pages (
        region TEXT, stem TEXT, prediction TEXT, seconds REAL, truncated INTEGER, error TEXT,
        PRIMARY KEY(region, stem))""")
    db.commit()
    done = set(db.execute("SELECT region, stem FROM pages WHERE error IS NULL"))
    pending = [(r,s) for r,s in region_pages if (r,s) not in done]
    if pending:
        read = load_reader(model, job["weights"], job["prompt"])
        def path(r, s):
            return Path(job["root"]) / r / "img" / (s + ".jpg")
        read(path(*pending[0]))
        for i, (r, s) in enumerate(pending):
            try:
                sync()
                start = time.perf_counter()
                prediction, truncated = read(path(r, s))
                sync()
                row = (r, s, prediction, time.perf_counter()-start, int(truncated), None)
            except Exception as exc:
                row = (r, s, None, None, None, repr(exc))
                print(r, s, repr(exc), flush=True)
            with db:
                db.execute("INSERT OR REPLACE INTO pages VALUES (?,?,?,?,?,?)", row)
            if (i+1) % 100 == 0:
                print(model, job["worker"], i+1, "/", len(pending), flush=True)
    db.close()

worker_code = ("import subprocess, sys, json\n" + inspect.getsource(load_reader) + "\n" +
               inspect.getsource(evaluate) + "\nevaluate(json.load(sys.stdin))\n")
from importlib.metadata import distributions
versions = {d.metadata["Name"].lower(): d.version for d in distributions() if d.metadata["Name"]}
workers = GPU_WORKERS if IS_GPU else 1
config = dict(model=MODEL, dataset=DATA_REV, revisions=revisions, weights_mib=weight_mib,
              prompt="", workers=workers, versions=versions,
              cpu=platform.machine(), threads=2, reader_sha256=hashlib.sha256(worker_code.encode()).hexdigest())
if Path("/proc/cpuinfo").exists():
    config["cpu"] = next((line.split(":",1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                          if line.startswith("model name")), config["cpu"])


positions = {page: i for i, page in enumerate(pages)}

def start_worker(index):
    job = dict(model=MODEL, pages=[page for page in selected if positions[page] % workers == index], gpu=IS_GPU, config=config,
               output=str(OUTPUT), root=str(ROOT), weights=weights, prompt="", worker=index)
    env = dict(runtime_env, USE_TF="0", USE_FLAX="0", CUDA_VISIBLE_DEVICES=str(index) if IS_GPU else "",
               OMP_NUM_THREADS="2", OMP_THREAD_LIMIT="2")
    # Forward logs explicitly; the timed worker/SQLite protocol stays unchanged.
    import tempfile
    with tempfile.TemporaryFile(mode="w+t", encoding="utf-8") as job_stream:
        job_stream.write(json.dumps(job))
        job_stream.seek(0)
        run_logged([str(model_python), "-I", "-u", "-c", worker_code], stdin=job_stream, env=env)
with ThreadPoolExecutor(max_workers=workers) as pool:
    list(pool.map(start_worker, range(workers)))

### Score saved predictions

Tables report page-level macro means. Check successful and expected page counts for each configuration before reporting full results.

In [ ]:
%%benchmark
from contextlib import closing
def occlusion_metadata(stem):
    import re
    parts = stem.split("__", 2)
    density = parts[1] if len(parts) > 1 and re.fullmatch(r"[0-9]+(?:p[0-9]+)?pct", parts[1]) else "unknown"
    return parts[0], density

def score_results():
    rows = []
    refs = {}
    for path in sorted(OUTPUT.glob(f"{MODEL}-*.sqlite")):
        with closing(sqlite3.connect(path)) as db:
            cfg, = db.execute("SELECT json FROM config").fetchone()
            config = json.loads(cfg)
            if config["dataset"] != DATA_REV:
                continue
            for region, stem, pred, seconds, trunc, error in db.execute("SELECT * FROM pages"):
                key = (region, stem)
                if key not in refs:
                    refs[key] = reference(*key)
                scores = (None,)*5 if error else metrics(refs[key], pred)
                parts = occlusion_metadata(stem)
                rows.append((config["model"], hashlib.sha256(cfg.encode()).hexdigest()[:12], config["device"],
                             config["workers"], config["weights_mib"], region, stem, parts[0], parts[1],
                             pred, refs[key], seconds, trunc, error, *scores))
    cols = ["model","config","device","workers","weights_mib","region","stem","type","density",
            "prediction","gt","seconds","truncated","error","cer","wer","precision","recall","f1"]
    return pd.DataFrame(rows, columns=cols)

results = score_results()
assert len(results) and not results.duplicated(["model","config","region","stem"]).any()
keys = ["model","config"]
values = ["cer","wer","precision","recall","f1","seconds"]
coverage = results.groupby(keys + ["region"]).agg(attempted=("stem","size"), ok=("seconds","count"),
                                                   truncated=("truncated","sum"))
coverage["expected"] = coverage.index.get_level_values("region").map(COUNTS)
display(coverage)
ok = results[results.error.isna()]
display(ok.groupby(keys+["device","weights_mib"])[values].mean())
for groups in (["region"], ["region","type"], ["region","density"]):
    display(ok.groupby(keys+groups)[values].mean().join(ok.groupby(keys+groups).size().rename("pages")))
latency = ok.groupby(keys+["workers","region"]).seconds.mean().unstack().reindex(columns=list(COUNTS))
display((latency.mul(pd.Series(COUNTS)).sum(axis=1, skipna=False) /
         (3600 * latency.index.get_level_values("workers"))).rename("projected_wall_hours"))
display(pd.DataFrame([(r, *occlusion_metadata(s)) for r,s in pages],
                     columns=["region","type","density"]).value_counts().rename("dataset_pages"))
# Locate qualitative candidates. Inspect the same image crop, GT and predictions before using them.
for region, group in ok.groupby("region"):
    print(region)
    display(pd.concat([group.nsmallest(1,"wer"),group.nlargest(1,"wer")])[
        ["model","stem","wer","gt","prediction"]])
# Example: display(Image.open(ROOT / region / "img" / (stem + ".jpg")))